# Extract Data From API

In [0]:
import requests
import pandas as pd

url="https://api.coingecko.com/api/v3/coins/markets"

params = {
  "vs_currency": "eur",
  "order": "market_cap_desc",
  "per_page": "100",
  "page": "1",
  "sparkline": "false"
}

response = requests.get(url, params=params)
data = response.json()

pd_df = pd.DataFrame(data)

df = spark.createDataFrame(pd_df)

display(df.limit(5))

# Store the raw Data to Bronze table

In [0]:
import pyspark.sql.functions as F

# Add ingestion metadata columns
df_final = df.withColumn(
    "_ingested_at", F.current_timestamp()
).withColumn(
    "_source", F.lit("coingecko_api")
)


df_final.write.mode("overwrite").format("delta").saveAsTable("crypto.bronze.brz_crypto")


In [0]:
%sql

SELECT * FROM crypto.bronze.brz_crypto
LIMIT 10